# VGGT 3D Reconstruction Walkthrough

This notebook recreates the 3D reconstruction steps used by `demo_viser.py` on the capture stored at `./dataset/astera_office_2/images/`. Each section exposes an intermediate prediction so you can inspect how VGGT builds cameras, depth, and point clouds.

## 0. Imports and runtime setup
Make sure the VGGT environment is activated before executing the cells.

In [4]:
import sys
sys.path.insert(0, 'G:/GithubProject/egocentric_control/vggt-loc/vggt')

In [5]:
%matplotlib inline

import os
from pathlib import Path
from contextlib import nullcontext

import numpy as np
import torch
import matplotlib.pyplot as plt
from PIL import Image

from vggt.models.vggt import VGGT
from vggt.utils.load_fn import load_and_preprocess_images
from vggt.utils.pose_enc import pose_encoding_to_extri_intri
from vggt.utils.geometry import unproject_depth_map_to_point_map, closed_form_inverse_se3


In [6]:
IMAGE_FOLDER = Path("astera_office_2/images")
assert IMAGE_FOLDER.exists(), f"Image folder not found: {IMAGE_FOLDER}"

image_paths = sorted(
    p for p in IMAGE_FOLDER.iterdir()
    if p.suffix.lower() in {".png", ".jpg", ".jpeg", ".bmp"}
)

MAX_IMAGES = None  # set to an integer to limit the sequence if memory becomes an issue
if MAX_IMAGES is not None:
    image_paths = image_paths[:MAX_IMAGES]

print(f"Found {len(image_paths)} images")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
dtype = torch.bfloat16 if device.type == "cuda" and torch.cuda.get_device_capability()[0] >= 8 else torch.float16
print(f"Using device: {device}, autocast dtype: {dtype}")


Found 41 images
Using device: cuda, autocast dtype: torch.bfloat16


In [ ]:
num_preview = min(4, len(image_paths))
if num_preview > 0:
    fig, axes = plt.subplots(1, num_preview, figsize=(4 * num_preview, 4))
    if num_preview == 1:
        axes = [axes]
    for ax, path in zip(axes, image_paths[:num_preview]):
        ax.imshow(Image.open(path))
        ax.set_title(path.name)
        ax.axis("off")
    plt.tight_layout()
else:
    raise RuntimeError("No images found for reconstruction.")


## 1. Load and preprocess images
This step mirrors `load_and_preprocess_images` from the demo script.


In [8]:
images_tensor = load_and_preprocess_images([str(p) for p in image_paths])
print(f"Preprocessed tensor shape: {images_tensor.shape}")
images_tensor = images_tensor.to(device)

Preprocessed tensor shape: torch.Size([41, 3, 294, 518])


## 2. Instantiate VGGT and load weights
Weights are fetched from the public Hugging Face checkpoint the first time you run this cell.

In [9]:
# model = VGGT()
# state_dict = torch.hub.load_state_dict_from_url("https://huggingface.co/facebook/VGGT-1B/resolve/main/model.pt", progress=True, map_location="cpu")
# model.load_state_dict(state_dict)
# model = model.to(device)
# model.eval()
# print("Model ready for inference.")
model = VGGT.from_pretrained("facebook/VGGT-1B").to(device)
model.eval();


## 3. Aggregate image tokens
VGGT first converts the image sequence into transformer tokens via the aggregator module.

In [10]:
with torch.no_grad():
    batched_images = images_tensor.unsqueeze(0)  # add batch dimension
    autocast_ctx = torch.cuda.amp.autocast(dtype=dtype) if device.type == "cuda" else nullcontext()
    with autocast_ctx:
        aggregated_tokens_list, patch_start_idx = model.aggregator(batched_images)

print(f"Number of refinement iterations: {len(aggregated_tokens_list)}")
print("Token shapes:", [t.shape for t in aggregated_tokens_list])
print(f"Patch start index: {patch_start_idx}")


G:\GithubProject/egocentric_control/vggt-loc/vggt\vggt\layers\attention.py:61: UserWarning: 1Torch was not compiled with flash attention. (Triggered internally at ..\aten\src\ATen\native\transformers\cuda\sdp_utils.cpp:455.)
  x = F.scaled_dot_product_attention(q, k, v, dropout_p=self.attn_drop.p if self.training else 0.0)


Number of refinement iterations: 24
Token shapes: [torch.Size([1, 41, 782, 2048]), torch.Size([1, 41, 782, 2048]), torch.Size([1, 41, 782, 2048]), torch.Size([1, 41, 782, 2048]), torch.Size([1, 41, 782, 2048]), torch.Size([1, 41, 782, 2048]), torch.Size([1, 41, 782, 2048]), torch.Size([1, 41, 782, 2048]), torch.Size([1, 41, 782, 2048]), torch.Size([1, 41, 782, 2048]), torch.Size([1, 41, 782, 2048]), torch.Size([1, 41, 782, 2048]), torch.Size([1, 41, 782, 2048]), torch.Size([1, 41, 782, 2048]), torch.Size([1, 41, 782, 2048]), torch.Size([1, 41, 782, 2048]), torch.Size([1, 41, 782, 2048]), torch.Size([1, 41, 782, 2048]), torch.Size([1, 41, 782, 2048]), torch.Size([1, 41, 782, 2048]), torch.Size([1, 41, 782, 2048]), torch.Size([1, 41, 782, 2048]), torch.Size([1, 41, 782, 2048]), torch.Size([1, 41, 782, 2048])]
Patch start index: 5


## 4. Predict cameras, depth, points, and tracks
Using the aggregated tokens we replicate the per-head calls from the demo script.

In [11]:
with torch.no_grad():
    # Camera poses
    pose_enc_list = model.camera_head(aggregated_tokens_list)
    pose_enc = pose_enc_list[-1]
    extrinsic, intrinsic = pose_encoding_to_extri_intri(pose_enc, batched_images.shape[-2:])

    # Dense depth and point predictions
    depth_map, depth_conf = model.depth_head(aggregated_tokens_list, batched_images, patch_start_idx)
    point_map, point_conf = model.point_head(aggregated_tokens_list, batched_images, patch_start_idx)

    # Example point tracking query
    query_points = torch.tensor([[100.0, 200.0], [60.72, 259.94]], device=device)
    track_list, vis_score, conf_score = model.track_head(
        aggregated_tokens_list, batched_images, patch_start_idx, query_points=query_points[None]
    )

print(f"Pose encoding shape: {pose_enc.shape}")
print(f"Extrinsic shape: {extrinsic.shape}, Intrinsic shape: {intrinsic.shape}")
print(f"Depth map shape: {depth_map.shape}, depth confidence shape: {depth_conf.shape}")
print(f"Point map shape: {point_map.shape}, point confidence shape: {point_conf.shape}")
print(f"Track list length: {len(track_list)}, track tensor shape: {track_list[-1].shape}")


Pose encoding shape: torch.Size([1, 41, 9])
Extrinsic shape: torch.Size([1, 41, 3, 4]), Intrinsic shape: torch.Size([1, 41, 3, 3])
Depth map shape: torch.Size([1, 41, 294, 518, 1]), depth confidence shape: torch.Size([1, 41, 294, 518])
Point map shape: torch.Size([1, 41, 294, 518, 3]), point confidence shape: torch.Size([1, 41, 294, 518])
Track list length: 4, track tensor shape: torch.Size([1, 41, 2, 2])


## 5. Convert depth to 3D points via camera geometry
Unprojecting the depth maps often yields cleaner 3D geometry than the point-map head.

In [12]:
point_map_by_unprojection = unproject_depth_map_to_point_map(
    depth_map.squeeze(0), extrinsic.squeeze(0), intrinsic.squeeze(0)
)
print(f"Unprojected point map shape: {point_map_by_unprojection.shape}")


Unprojected point map shape: (41, 294, 518, 3)


## 6. Visualize intermediate predictions

In [ ]:
frame_idx = 0  # change to visualize a different frame

fig, axs = plt.subplots(1, 2, figsize=(10, 4))
axs[0].imshow(images_tensor[frame_idx].detach().cpu().permute(1, 2, 0))
axs[0].set_title(f"Input frame {frame_idx}")
axs[0].axis("off")

depth_vis = depth_map[0, frame_idx].detach().cpu().squeeze().numpy()
axs[1].imshow(depth_vis, cmap="magma")
axs[1].set_title("Predicted depth")
axs[1].axis("off")
plt.tight_layout()


In [14]:
import open3d as o3d

sample_frame = 1
points = point_map_by_unprojection[sample_frame].reshape(-1, 3)
colors = images_tensor[sample_frame].detach().cpu().permute(1, 2, 0).reshape(-1, 3).numpy()

# Create Open3D point cloud
pcd = o3d.geometry.PointCloud()
pcd.points = o3d.utility.Vector3dVector(points)
pcd.colors = o3d.utility.Vector3dVector(colors)

# Visualize
print(f"Visualizing {len(pcd.points)} points")
o3d.visualization.draw_geometries(
    [pcd],
    window_name="3D Point Cloud (World Coordinates)",
    width=1024,
    height=768,
    left=50,
    top=50,
    point_show_normal=False
)

Visualizing 152292 points


In [15]:
import open3d as o3d

# Combine all frames into one point cloud
all_points = []
all_colors = []

for frame_idx in range(len(images_tensor)):
    points = point_map_by_unprojection[frame_idx].reshape(-1, 3)
    colors = images_tensor[frame_idx].detach().cpu().permute(1, 2, 0).reshape(-1, 3).numpy()
    all_points.append(points)
    all_colors.append(colors)

all_points = np.vstack(all_points)
all_colors = np.vstack(all_colors)

# Create and visualize combined point cloud
pcd = o3d.geometry.PointCloud()
pcd.points = o3d.utility.Vector3dVector(all_points)
pcd.colors = o3d.utility.Vector3dVector(all_colors)

print(f"Visualizing {len(pcd.points)} points from {len(images_tensor)} frames")
o3d.visualization.draw_geometries(
    [pcd],
    window_name="Full Scene Reconstruction (All Frames)",
    width=1024,
    height=768,
    left=50,
    top=50,
    point_show_normal=False
)

Visualizing 6243972 points from 41 frames


In [16]:
# Combine all frames with confidence filtering (matching demo_viser.py behavior)
all_points = []
all_colors = []
all_conf = []

for frame_idx in range(len(images_tensor)):
    points = point_map_by_unprojection[frame_idx].reshape(-1, 3)
    colors = images_tensor[frame_idx].detach().cpu().permute(1, 2, 0).reshape(-1, 3).numpy()
    conf = depth_conf[0, frame_idx].detach().cpu().numpy().reshape(-1)
    
    all_points.append(points)
    all_colors.append(colors)
    all_conf.append(conf)

# Flatten all frames
points_flat = np.vstack(all_points)
colors_flat = np.vstack(all_colors)
conf_flat = np.concatenate(all_conf)

# Filter by confidence (matching demo_viser.py logic)
percentile = 25.0  # Default --conf_threshold, adjust to remove more/less noise
cutoff = np.percentile(conf_flat, percentile)
valid = (conf_flat >= cutoff) & (conf_flat > 1e-5)

print(f"Confidence cutoff at {percentile}th percentile: {cutoff:.6f}")
print(f"Keeping {valid.sum():,} / {len(valid):,} points ({100 * valid.sum() / len(valid):.1f}%)")

# Filter points and colors
points_filtered = points_flat[valid]
colors_filtered = colors_flat[valid]

# Optional: center the point cloud like demo_viser.py
points_centered = points_filtered - points_filtered.mean(axis=0)

# Create Open3D point cloud
pcd = o3d.geometry.PointCloud()
pcd.points = o3d.utility.Vector3dVector(points_centered)  # Use points_filtered if you don't want centering
pcd.colors = o3d.utility.Vector3dVector(colors_filtered)

# Optional: statistical outlier removal for even cleaner results
pcd, outlier_mask = pcd.remove_statistical_outlier(nb_neighbors=30, std_ratio=1.0)
print(f"After outlier removal: {len(pcd.points):,} points")

# Visualize
print(f"Visualizing filtered point cloud from {len(images_tensor)} frames")
o3d.visualization.draw_geometries(
    [pcd],
    window_name="Filtered Scene Reconstruction (All Frames)",
    width=1024,
    height=768,
    left=50,
    top=50,
    point_show_normal=False
)

Confidence cutoff at 25.0th percentile: 1.000820
Keeping 4,682,990 / 6,243,972 points (75.0%)
After outlier removal: 4,150,105 points
Visualizing filtered point cloud from 41 frames


## 7. Visualize camera poses
Overlay the estimated camera frames on an Open3D rendering of the filtered point cloud (OpenCV convention: x→red, y→green, z→blue).

In [17]:
import open3d as o3d
from vggt.utils.geometry import closed_form_inverse_se3

# Confidence-filtered point cloud (match demo_viser percentile filtering)
conf_flat = depth_conf[0].detach().cpu().numpy().reshape(-1)
points_flat = point_map_by_unprojection.reshape(-1, 3)
colors_flat = images_tensor.detach().cpu().permute(0, 2, 3, 1).numpy().reshape(-1, 3)

percentile = 25.0  # drop the lowest-confidence pixels
cutoff = np.percentile(conf_flat, percentile)
valid = (conf_flat >= cutoff) & (conf_flat > 1e-5)

pcd_filtered = o3d.geometry.PointCloud()
pcd_filtered.points = o3d.utility.Vector3dVector(points_flat[valid])
pcd_filtered.colors = o3d.utility.Vector3dVector(colors_flat[valid])

# Rotate scene so that Y becomes up (180° about X to preserve handedness)
world_fix = np.array(
    [
        [1.0, 0.0, 0.0, 0.0],
        [0.0, -1.0, 0.0, 0.0],
        [0.0, 0.0, -1.0, 0.0],
        [0.0, 0.0, 0.0, 1.0],
    ],
    dtype=np.float32,
)

pcd_filtered.transform(world_fix)

# Build camera frames in world coordinates
extrinsic_batch = extrinsic.squeeze(0).detach()
S = extrinsic_batch.shape[0]
extrinsic_h = torch.zeros((S, 4, 4), dtype=extrinsic_batch.dtype, device=extrinsic_batch.device)
extrinsic_h[:, :3, :4] = extrinsic_batch
extrinsic_h[:, 3, 3] = 1.0

cam_to_world = closed_form_inverse_se3(extrinsic_h).cpu().numpy()

frame_scale = 0.05  # adjust to match your scene scale
camera_frames = []
for idx, T in enumerate(cam_to_world):
    frame = o3d.geometry.TriangleMesh.create_coordinate_frame(size=frame_scale)
    frame.transform(world_fix @ T)
    camera_frames.append(frame)

print(f"Filtered points: {len(pcd_filtered.points)} | Cameras: {len(camera_frames)}")

o3d.visualization.draw_geometries(
    [pcd_filtered, *camera_frames],
    window_name="Point cloud with camera poses",
    width=1024,
    height=768,
    left=100,
    top=100,
)


Filtered points: 4682990 | Cameras: 41


### 7.1 ICP Alignment 

In [50]:
import copy
import math
import time
from typing import Iterable, Tuple, List, Optional, Union

import numpy as np
import open3d as o3d


# ---------- helpers that mirror your script's approach ----------

def _sample_points(points: np.ndarray, max_points: int, rng: np.random.Generator) -> np.ndarray:
    if points.shape[0] <= max_points:
        return points.copy()
    idx = rng.choice(points.shape[0], size=max_points, replace=False)
    return points[idx]


def _estimate_threshold(points_a: np.ndarray, points_b: np.ndarray) -> float:
    def bbox_diag(pts: np.ndarray) -> float:
        if pts.shape[0] == 0:
            return 0.0
        extent = np.max(pts, axis=0) - np.min(pts, axis=0)
        return float(np.linalg.norm(extent))

    diag = max(bbox_diag(points_a), bbox_diag(points_b))
    return max(1e-4, diag * 0.05)


def _to_numpy_points(pcd: o3d.geometry.PointCloud) -> np.ndarray:
    return np.asarray(pcd.points, dtype=np.float64)


def _is_pose_matrix(x) -> bool:
    return isinstance(x, np.ndarray) and x.shape == (4, 4)


def _as_iter(x):
    if x is None:
        return []
    if isinstance(x, (list, tuple)):
        return list(x)
    return [x]


def _rotation_angle_deg(R: np.ndarray) -> float:
    """Return rotation angle (degrees) from a 3x3 rotation matrix."""
    # Clamp for numerical stability
    trace_term = (np.trace(R) - 1.0) / 2.0
    trace_term = float(np.clip(trace_term, -1.0, 1.0))
    return math.degrees(math.acos(trace_term))


# ---------- main API ----------

def icp_register(
    reference_pcd: o3d.geometry.PointCloud,
    source_pcd: o3d.geometry.PointCloud,
    source_camera: Union[
        None,
        o3d.geometry.Geometry3D,
        List[o3d.geometry.Geometry3D],
        np.ndarray,
        List[np.ndarray],
    ],
    preview: bool = False,
    sample_size: int = 4000,
    max_iterations: int = 30,
    tolerance: float = 1e-5,
    rng: Optional[np.random.Generator] = None,
) -> Tuple[o3d.geometry.PointCloud, Union[None, o3d.geometry.Geometry3D, List[o3d.geometry.Geometry3D], np.ndarray, List[np.ndarray]], np.ndarray]:
    """
    Align `source_pcd` to `reference_pcd` using Open3D ICP (point-to-point), mirroring the
    method in your CLI script (sampling, bbox-based threshold, relative criteria).
    Cameras are not used in ICP; they are transformed afterward by the resulting T.

    Args:
        reference_pcd: Target cloud (Open3D PointCloud).
        source_pcd: Source cloud (Open3D PointCloud).
        source_camera: Either:
            • Open3D geometry (or list of geometries), OR
            • 4x4 numpy pose matrix (or list of such matrices).
          Output camera type matches input type.
        preview: Show ref (blue), src-before (red), src-after (green),
                 cameras before (cyan) and after (yellow).
        sample_size: Max points sampled from each cloud for ICP (like your script).
        max_iterations: ICP max iterations.
        tolerance: Relative fitness/RMSE tolerances (same fields as your script).
        rng: Optional np.random.Generator; if None, a default seeded generator is used.

    Returns:
        transformed_pcd: `source_pcd` transformed by ICP (new object).
        transformed_camera: Cameras transformed by T (same type as input).
        T: 4x4 numpy array, transform mapping source -> reference.
    """

    start_time=time.time()
    if rng is None:
        rng = np.random.default_rng(42)

    # --- prepare sampled point sets for ICP (like your script) ---
    ref_pts_full = _to_numpy_points(reference_pcd)
    src_pts_full = _to_numpy_points(source_pcd)

    if ref_pts_full.size == 0 or src_pts_full.size == 0:
        raise ValueError("Empty point cloud(s). Cannot run ICP.")

    ref_pts = _sample_points(ref_pts_full, sample_size, rng)
    src_pts = _sample_points(src_pts_full, sample_size, rng)

    # Distance threshold from bbox diagonal (script-style)
    distance_threshold = _estimate_threshold(src_pts, ref_pts)

    # Build temporary Open3D clouds for ICP
    src_cloud = o3d.geometry.PointCloud()
    src_cloud.points = o3d.utility.Vector3dVector(src_pts)
    ref_cloud = o3d.geometry.PointCloud()
    ref_cloud.points = o3d.utility.Vector3dVector(ref_pts)

    estimation = o3d.pipelines.registration.TransformationEstimationPointToPoint()
    criteria = o3d.pipelines.registration.ICPConvergenceCriteria(
        max_iteration=max_iterations,
        relative_fitness=tolerance,
        relative_rmse=tolerance,
    )

    icp = o3d.pipelines.registration.registration_icp(
        src_cloud,
        ref_cloud,
        distance_threshold,
        np.eye(4, dtype=np.float64),
        estimation,
        criteria,
    )

    T = np.asarray(icp.transformation, dtype=np.float64)  # source -> reference

    # ---- print rotation (deg) and translation norm ----
    rot_deg = _rotation_angle_deg(T[:3, :3])
    trans_norm = float(np.linalg.norm(T[:3, 3]))
    print(f"ICP rotation (deg): {rot_deg:.4f}")
    print(f"ICP translation norm: {trans_norm:.6f}")

    # --- apply T to the full-resolution source cloud (not just sampled) ---
    transformed_pcd = copy.deepcopy(source_pcd)
    transformed_pcd.transform(T)

    # --- apply T to cameras (AFTER ICP), preserving input type ---
    cams_in = _as_iter(source_camera)
    transformed_camera = None
    if len(cams_in) == 0:
        cam_before_iter = []
        cam_after_iter = []
    elif all(_is_pose_matrix(c) for c in cams_in):
        transformed_camera = [T @ c for c in cams_in]
        cam_before_iter = cams_in
        cam_after_iter = transformed_camera
    elif all(not _is_pose_matrix(c) for c in cams_in):
        def _tx_geom(g):
            g2 = copy.deepcopy(g)
            g2.transform(T)
            return g2
        transformed_camera = [_tx_geom(g) for g in cams_in]
        cam_before_iter = cams_in
        cam_after_iter = transformed_camera
    else:
        raise TypeError("source_camera must be all 4x4 pose matrices OR all Open3D geometries.")

    # collapse single item if single input
    if isinstance(source_camera, (list, tuple)):
        pass
    elif source_camera is None:
        transformed_camera = None
    else:
        transformed_camera = transformed_camera[0]
    
    # Log time taken
    end_time=time.time()
    print(f"ICP registration took: {end_time - start_time:.2f} seconds")
    
    # --- optional preview (match your preview color scheme + camera colors) ---
    if preview:
        # Colors: before=red, after=green, reference=blue; cams before=cyan, after=yellow
        BEFORE_COLOR = (1.0, 0.0, 0.0)
        AFTER_COLOR  = (0.0, 0.8, 0.0)
        REF_COLOR    = (0.0, 0.4, 1.0)
        CAM_BEFORE   = (0.0, 1.0, 1.0)
        CAM_AFTER    = (1.0, 1.0, 0.0)
        print("Colors:  src: red, trans: green, reference: blue, cameras cyan → yellow")

        # Build colored copies
        ref_vis = reference_pcd.clone() if hasattr(reference_pcd, "clone") else copy.deepcopy(reference_pcd)
        src_vis = source_pcd.clone() if hasattr(source_pcd, "clone") else copy.deepcopy(source_pcd)
        out_vis = transformed_pcd.clone() if hasattr(transformed_pcd, "clone") else copy.deepcopy(transformed_pcd)

        ref_vis.paint_uniform_color(REF_COLOR)
        src_vis.paint_uniform_color(BEFORE_COLOR)
        out_vis.paint_uniform_color(AFTER_COLOR)

        vis_geoms: List[o3d.geometry.Geometry3D] = [ref_vis, src_vis, out_vis]

        def _pose_to_frame(M: np.ndarray, size: float = 0.05) -> o3d.geometry.TriangleMesh:
            fr = o3d.geometry.TriangleMesh.create_coordinate_frame(size=size)
            fr.transform(M)
            return fr

        if len(cams_in) > 0 and all(_is_pose_matrix(c) for c in cams_in):
            cam_before_frames = [_pose_to_frame(M) for M in cam_before_iter]
            cam_after_frames  = [_pose_to_frame(M) for M in cam_after_iter]
            for g in cam_before_frames:
                try: g.paint_uniform_color(CAM_BEFORE)
                except Exception: pass
            for g in cam_after_frames:
                try: g.paint_uniform_color(CAM_AFTER)
                except Exception: pass
            vis_geoms += cam_before_frames + cam_after_frames
        else:
            cam_before_col = []
            for g in cam_before_iter:
                gc = copy.deepcopy(g)
                try: gc.paint_uniform_color(CAM_BEFORE)
                except Exception: pass
                cam_before_col.append(gc)
            cam_after_col = []
            for g in cam_after_iter:
                gc = copy.deepcopy(g)
                try: gc.paint_uniform_color(CAM_AFTER)
                except Exception: pass
                cam_after_col.append(gc)
            vis_geoms += cam_before_col + cam_after_col

        o3d.visualization.draw_geometries(
            vis_geoms,
            window_name="ICP preview [before: red | after: green | ref: blue | cams: cyan→yellow]",
            width=1280, height=800, left=100, top=100
        )

    return transformed_pcd, transformed_camera, T


#### 7.1.1 Test ICP samples

In [40]:
example_reference_ply = r"G:\GithubProject\egocentric_control\vggt-loc\results\result\point_cloud.ply"
example_src_ply       = r"G:\GithubProject\egocentric_control\vggt-loc\results\result4\point_cloud.ply"
example_src_camera_frames = r"G:\GithubProject\egocentric_control\vggt-loc\results\result4\all_camera_poses.txt"


In [35]:
import numpy as np
import open3d as o3d
from typing import List, Tuple


def _parse_camera_poses_txt(txt_path: str) -> List[np.ndarray]:
    """
    Parse a text file containing multiple 4x4 camera-to-world matrices.
    Lines beginning with '#' or blank lines are ignored.
    Matrices are read as contiguous groups of 4 numeric rows (16 numbers each).
    Returns a list of (4,4) float64 numpy arrays.
    """
    mats = []
    buf = []

    def _flush_buf():
        nonlocal buf, mats
        if len(buf) == 4:
            M = np.stack(buf, axis=0).astype(np.float64)
            if M.shape == (4, 4):
                mats.append(M)
        buf = []

    with open(txt_path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line or line.startswith("#"):
                # header/comment/blank: separator — if we had 4 rows collected, flush
                _flush_buf()
                continue
            # try parse a numeric row
            parts = line.split()
            # accept rows that look like 4 numbers; ignore anything else
            try:
                row = [float(x) for x in parts]
            except ValueError:
                # non-numeric line: also a separator
                _flush_buf()
                continue
            if len(row) == 4:
                buf.append(np.array(row, dtype=np.float64))
                if len(buf) == 4:
                    _flush_buf()
            else:
                # not a 4-column numeric row; treat as separator
                _flush_buf()

    # in case file ends exactly after 4 rows or with trailing rows
    _flush_buf()
    return mats


def load_icp_inputs(
    reference_ply_path: str,
    source_ply_path: str,
    camera_txt_path: str,
    frame_scale: float = 0.05,
    apply_world_fix: bool = False,
) -> Tuple[o3d.geometry.PointCloud, o3d.geometry.PointCloud, List[o3d.geometry.Geometry3D]]:
    """
    Load reference/source point clouds from .ply and camera frames from a .txt of 4x4 matrices.

    Args:
        reference_ply_path: Path to reference point cloud (.ply).
        source_ply_path: Path to source point cloud (.ply).
        camera_txt_path: Path to text file with camera-to-world 4x4 matrices.
        frame_scale: Size for Open3D coordinate frames created for each camera.
        apply_world_fix: If True, applies a 180° rotation about X (Y-up) to both PCDs and cameras.

    Returns:
        ref_pcd: Open3D PointCloud (reference).
        src_pcd: Open3D PointCloud (source).
        src_cameras: List of Open3D geometries (coordinate frames) in world coordinates.
    """
    # Load point clouds
    ref_pcd = o3d.io.read_point_cloud(reference_ply_path)
    src_pcd = o3d.io.read_point_cloud(source_ply_path)

    if ref_pcd.is_empty():
        raise ValueError(f"Reference PLY has no points: {reference_ply_path}")
    if src_pcd.is_empty():
        raise ValueError(f"Source PLY has no points: {source_ply_path}")

    # Optional world fix (match your earlier convention: Y-up by 180° about X)
    world_fix = np.array(
        [[1.0,  0.0,  0.0, 0.0],
         [0.0, -1.0,  0.0, 0.0],
         [0.0,  0.0, -1.0, 0.0],
         [0.0,  0.0,  0.0, 1.0]], dtype=np.float64
    )

    if apply_world_fix:
        ref_pcd = ref_pcd.transform(world_fix.copy())
        # transform returns None in-place; we want copies, so re-load then transform
        ref_pcd = o3d.io.read_point_cloud(reference_ply_path)
        ref_pcd.transform(world_fix.copy())

        src_pcd = o3d.io.read_point_cloud(source_ply_path)
        src_pcd.transform(world_fix.copy())

    # Parse camera-to-world matrices and build frames
    cam2world_list = _parse_camera_poses_txt(camera_txt_path)

    src_cameras: List[o3d.geometry.Geometry3D] = []
    for T in cam2world_list:
        frame = o3d.geometry.TriangleMesh.create_coordinate_frame(size=frame_scale)
        if apply_world_fix:
            frame.transform(world_fix @ T)
        else:
            frame.transform(T)
        src_cameras.append(frame)

    return ref_pcd, src_pcd, src_cameras


In [41]:
# Load the three objects your ICP function expects
ref_pcd, src_pcd, src_cameras = load_icp_inputs(
    reference_ply_path=example_reference_ply,
    source_ply_path=example_src_ply,
    camera_txt_path=example_src_camera_frames,
    frame_scale=0.05,
    apply_world_fix=False,  # set True if you used the same 180° X-rotation elsewhere
)
print(ref_pcd, src_pcd, f"{len(src_cameras)} camera frames loaded")

PointCloud with 8292034 points. PointCloud with 8277497 points. 45 camera frames loaded


In [51]:


aligned_pcd, aligned_cameras, T = icp_register(
    reference_pcd=ref_pcd,
    source_pcd=src_pcd,
    source_camera=src_cameras,
    preview=True,  # set False to skip visualization
)


print("Estimated transformation (source -> reference):")
print(T)

ICP rotation (deg): 0.3026
ICP translation norm: 0.011154
ICP registration took: 0.18 seconds
Colors:  src: red, trans: green, reference: blue, cameras cyan → yellow
Estimated transformation (source -> reference):
[[ 9.99988245e-01 -4.38655819e-03 -2.06603416e-03  3.88227534e-04]
 [ 4.38222260e-03  9.99988196e-01 -2.09838210e-03  5.68171067e-03]
 [ 2.07521445e-03  2.08930361e-03  9.99995664e-01  9.59111182e-03]
 [ 0.00000000e+00  0.00000000e+00  0.00000000e+00  1.00000000e+00]]


### 7.2 Ground finding 

In [32]:
import numpy as np
import open3d as o3d
import time
from typing import List, Tuple, Dict, Any

def find_ground_and_camera_planes(
    pcd_filtered: o3d.geometry.PointCloud,
    camera_frames: List[o3d.geometry.TriangleMesh],
    *,
    show_viewer: bool = False,
    ground_color: Tuple[float, float, float] = (0.8, 0.4, 0.2),
    camera_plane_color: Tuple[float, float, float] = (0.0, 0.8, 0.3),
    ransac_dist_thresh: float = 0.01,
    ransac_max_iters: int = 1500,
    window_name: str = "Point cloud with ground & camera planes",
    width: int = 1280,
    height: int = 720
) -> Dict[str, Any]:
    """
    Finds BOTH planes:
      - Ground plane: RANSAC on point cloud
      - Camera plane: least-squares fit on camera frame centers
    Prints timing for each, and (optionally) visualizes.
    Returns ONLY the plane definitions: normal and d (n·x + d = 0).
    """

    assert isinstance(pcd_filtered, o3d.geometry.PointCloud), "pcd_filtered must be an Open3D PointCloud"
    if len(pcd_filtered.points) == 0:
        raise ValueError("pcd_filtered has no points.")
    if camera_frames is None or len(camera_frames) < 3:
        raise ValueError("Need at least 3 camera frames to fit the camera plane.")

    # ---------- helpers ----------
    def ransac_plane(points: np.ndarray, dist_thresh: float, max_iters: int):
        best_inliers = None
        best_normal = None
        best_d = None
        best_count = 0
        n = points.shape[0]
        if n < 3:
            return None, None, None

        rng = np.random.default_rng()
        for _ in range(max_iters):
            idx = rng.choice(n, 3, replace=False)
            p1, p2, p3 = points[idx]
            v1 = p2 - p1
            v2 = p3 - p1
            normal = np.cross(v1, v2)
            norm = np.linalg.norm(normal)
            if norm < 1e-9:
                continue
            normal = normal / norm
            d = -np.dot(normal, p1)
            distances = np.abs(points @ normal + d)
            inliers = distances < dist_thresh
            count = int(inliers.sum())
            if count > best_count:
                best_count = count
                best_inliers = inliers
                best_normal = normal
                best_d = d

        # refine with least squares on inliers
        if best_inliers is not None and best_count >= 3:
            inlier_pts = points[best_inliers]
            ctr = inlier_pts.mean(axis=0)
            centered = inlier_pts - ctr
            _, _, vh = np.linalg.svd(centered, full_matrices=False)
            normal = vh[-1]
            normal = normal / np.linalg.norm(normal)
            d = -np.dot(normal, ctr)
            return normal, d, best_inliers
        return None, None, None

    def plane_local_coords(points3d: np.ndarray, normal: np.ndarray, d: float):
        # Project to plane and build orthonormal basis (u,v) on the plane
        distances = points3d @ normal + d
        proj = points3d - np.outer(distances, normal)

        arbitrary = np.array([1.0, 0.0, 0.0]) if abs(normal[0]) < 0.9 else np.array([0.0, 1.0, 0.0])
        u = arbitrary - (arbitrary @ normal) * normal
        u /= np.linalg.norm(u)
        v = np.cross(normal, u)
        v /= np.linalg.norm(v)

        origin = proj.mean(axis=0)
        rel = proj - origin
        coords_2d = np.stack([rel @ u, rel @ v], axis=1)
        return coords_2d, u, v, origin

    def rect_mesh_from_plane_bbox(normal: np.ndarray, d: float, points3d: np.ndarray,
                                  color=(0.8, 0.4, 0.2)):
        coords_2d, u, v, origin = plane_local_coords(points3d, normal, d)
        umin, umax = coords_2d[:, 0].min(), coords_2d[:, 0].max()
        vmin, vmax = coords_2d[:, 1].min(), coords_2d[:, 1].max()
        corners_2d = np.array([[umin, vmin], [umax, vmin], [umax, vmax], [umin, vmax]])
        verts = np.array([origin + u*u2 + v*v2 for (u2, v2) in corners_2d])

        tri_idx = np.array([[0, 1, 2], [0, 2, 3], [0, 2, 1], [0, 3, 2]], dtype=np.int32)  # double-sided
        mesh = o3d.geometry.TriangleMesh()
        mesh.vertices = o3d.utility.Vector3dVector(verts)
        mesh.triangles = o3d.utility.Vector3iVector(tri_idx)
        mesh.compute_vertex_normals()
        mesh.paint_uniform_color(color)
        return mesh

    def least_squares_plane(points3d: np.ndarray):
        ctr = points3d.mean(axis=0)
        centered = points3d - ctr
        _, _, vh = np.linalg.svd(centered, full_matrices=False)
        n = vh[-1]
        n /= np.linalg.norm(n)
        d = -np.dot(n, ctr)
        return n, d

    def camera_centers_from_frames(meshes: List[o3d.geometry.TriangleMesh]) -> np.ndarray:
        # Approximate each frame's center as its AABB center (good for transformed coord frames)
        centers = []
        for m in meshes:
            aabb = m.get_axis_aligned_bounding_box()
            centers.append(np.asarray(aabb.get_center()))
        return np.array(centers)

    # ---------- ground plane via RANSAC (timed) ----------
    pts = np.asarray(pcd_filtered.points)

    t0 = time.perf_counter()
    g_normal, g_d, _ = ransac_plane(pts, ransac_dist_thresh, ransac_max_iters)
    t1 = time.perf_counter()
    if g_normal is None:
        raise RuntimeError("RANSAC failed to find a ground plane.")
    print(f"[Timing] Ground plane finding: {t1 - t0:.4f} s")

    # ---------- camera plane via LS on camera centers (timed) ----------
    cam_centers = camera_centers_from_frames(camera_frames)
    if cam_centers.shape[0] < 3:
        raise RuntimeError("Not enough camera frames to fit a camera plane (need >= 3).")
    t2 = time.perf_counter()
    c_normal, c_d = least_squares_plane(cam_centers)
    t3 = time.perf_counter()
    print(f"[Timing] Camera plane finding: {t3 - t2:.4f} s")

    # ---------- optional visualization ----------
    if show_viewer:
        ground_mesh = rect_mesh_from_plane_bbox(g_normal, g_d, pts, color=ground_color)
        camera_mesh = rect_mesh_from_plane_bbox(c_normal, c_d, pts, color=camera_plane_color)
        geoms = [pcd_filtered, ground_mesh, camera_mesh, *camera_frames]
        o3d.visualization.draw_geometries(
            geoms,
            window_name=window_name,
            width=width,
            height=height,
            left=80,
            top=60,
        )

    # ---------- simplified return (only plane definitions) ----------
    return {
        "ground": {"normal": g_normal, "d": float(g_d)},
        "camera_plane": {"normal": c_normal, "d": float(c_d)},
    }


In [20]:
# Assuming you already have: pcd_filtered (PointCloud) and camera_frames (list of meshes)
res = find_ground_and_camera_planes(pcd_filtered, camera_frames, show_viewer=True, ransac_max_iters=1000)
print("Ground plane:", res["ground"])
print("Camera plane:", res["camera_plane"])



[Timing] Ground plane finding: 30.5103 s
[Timing] Camera plane finding: 0.0001 s
Ground plane: {'normal': array([-0.01178992, -0.98100589, -0.19361933]), 'd': -0.14950013383827704}
Camera plane: {'normal': array([0.02819322, 0.97994336, 0.19727177]), 'd': 0.026561948621765197}


### 7.3 Resgitration

In [21]:
import numpy as np
import open3d as o3d
from typing import List, Dict, Any

def register_to_ground_z0(
    pcd_filtered: o3d.geometry.PointCloud,
    camera_frames: List[o3d.geometry.TriangleMesh],
    ground_normal: np.ndarray,
    ground_d: float,
    *,
    show_viewer: bool = False,
    window_name: str = "Registered to z=0 (ground)",
    width: int = 1280,
    height: int = 720
) -> Dict[str, Any]:
    """
    Rigidly register a point cloud and camera frames so that the given ground plane
    (n · x + d = 0) becomes the plane z = 0, with the majority of points at z > 0.

    Returns:
        {
          "pcd": <registered PointCloud>,
          "camera_frames": <list of registered meshes>,
          "T_input_to_registered": <4x4 numpy array>
        }
    """

    assert isinstance(pcd_filtered, o3d.geometry.PointCloud), "pcd_filtered must be an Open3D PointCloud"
    assert len(pcd_filtered.points) > 0, "pcd_filtered has no points"
    assert ground_normal is not None and np.linalg.norm(ground_normal) > 0, "ground_normal must be nonzero"
    assert isinstance(camera_frames, list), "camera_frames must be a list (can be empty)"

    # Normalize plane normal
    n = np.asarray(ground_normal, dtype=float)
    n = n / np.linalg.norm(n)
    d = float(ground_d)

    pts = np.asarray(pcd_filtered.points)

    # Ensure "up" means z > 0 after registration:
    # If the average signed distance (n·x + d) is negative, flip the plane.
    signed = pts @ n + d
    if signed.mean() < 0.0:
        n = -n
        d = -d

    # --- Build rotation R that maps n -> ez ---
    ez = np.array([0.0, 0.0, 1.0], dtype=float)
    v = np.cross(n, ez)
    s = np.linalg.norm(v)
    c = float(np.dot(n, ez))

    if s < 1e-12:
        # n already aligned with +Z or -Z after the flip logic above
        if c > 0:  # n == +ez
            R = np.eye(3)
        else:      # n == -ez (shouldn't happen after flip, but handle anyway)
            # 180° around any axis in the plane; choose X
            R = np.array([[1,0,0],[0,-1,0],[0,0,-1]], dtype=float)
    else:
        # Rodrigues' rotation formula: R = I + K + K^2 * ((1 - c) / s^2), where K = [v]_x
        kx, ky, kz = v / s  # unit rotation axis
        K = np.array([[0, -kz, ky],
                      [kz, 0, -kx],
                      [-ky, kx, 0]], dtype=float)
        R = np.eye(3) + K * s + (K @ K) * ((1.0 - c) / (s * s))

    # After rotation, z' = n·x (because R maps n -> ez). The plane becomes z' + d = 0.
    # So translate by t = [0, 0, d] to move the plane to z=0.
    t = np.array([0.0, 0.0, d], dtype=float)

    # 4x4 transform T: x_new = R x + t
    T = np.eye(4, dtype=float)
    T[:3, :3] = R
    T[:3, 3] = t

    # Apply to copies (don’t mutate inputs)
    reg_pcd = o3d.geometry.PointCloud(pcd_filtered)  # shallow-copy ctor copies data refs; .transform makes its own
    reg_pcd.transform(T)

    reg_cams = []
    for m in camera_frames:
        mc = o3d.geometry.TriangleMesh(m)  # copy
        mc.transform(T)
        reg_cams.append(mc)

    # Optional visualization
    if show_viewer:
        geoms = [reg_pcd, *reg_cams]
        # Add an origin coordinate frame indicator in the registered space
        origin_frame = o3d.geometry.TriangleMesh.create_coordinate_frame(size=0.25, origin=[0, 0, 0])
        geoms.append(origin_frame)

        o3d.visualization.draw_geometries(
            geoms,
            window_name=window_name,
            width=width,
            height=height,
            left=80,
            top=60,
        )

    return {
        "pcd": reg_pcd,
        "camera_frames": reg_cams,
        "T_input_to_registered": T,
    }


In [22]:
regis_res= register_to_ground_z0(
    pcd_filtered,
    camera_frames,
    ground_normal=res["ground"]["normal"],
    ground_d=res["ground"]["d"],
    show_viewer=True
)

print(regis_res["T_input_to_registered"])

[[ 0.99987901 -0.01006725 -0.01178992  0.        ]
 [-0.01006725  0.16233295 -0.98100589  0.        ]
 [ 0.01178992  0.98100589  0.16221196  0.14950013]
 [ 0.          0.          0.          1.        ]]


## 8. Derive floor normal and top-down projection
Average the camera orientations to estimate the floor normal, rotate the filtered 3D points into that frame, and rasterize a bird's-eye image.

In [ ]:
from vggt.utils.geometry import closed_form_inverse_se3

# Confidence-filtered points (match demo_viser percentile filtering)
conf_flat = depth_conf[0].detach().cpu().numpy().reshape(-1)
points_flat = point_map_by_unprojection.reshape(-1, 3)

percentile = 25.0
cutoff = np.percentile(conf_flat, percentile)
valid_mask = (conf_flat >= cutoff) & (conf_flat > 1e-5)
points_valid = points_flat[valid_mask]
print(f"Using {points_valid.shape[0]} high-confidence points for top-down density map")

# Estimate floor normal from camera extrinsics (OpenCV convention: +Y points down)
R_world_to_cam = extrinsic.squeeze(0).detach().cpu().numpy()[:, :3, :3]

down_dirs = R_world_to_cam[:, 1, :].copy()
ref_down = down_dirs[0]
for i in range(len(down_dirs)):
    if np.dot(down_dirs[i], ref_down) < 0:
        down_dirs[i] *= -1
avg_down = down_dirs.mean(axis=0)
floor_normal = -avg_down / np.linalg.norm(avg_down)
print(f"Estimated floor normal (world frame): {floor_normal}")

# Build horizontal basis vectors
x_dirs = R_world_to_cam[:, 0, :].copy()
for i in range(len(x_dirs)):
    if np.dot(x_dirs[i], x_dirs[0]) < 0:
        x_dirs[i] *= -1
x_dirs_proj = x_dirs - (x_dirs @ floor_normal)[:, None] * floor_normal
basis_x = x_dirs_proj.mean(axis=0)
basis_x /= np.linalg.norm(basis_x)

basis_y = np.cross(floor_normal, basis_x)
basis_y /= np.linalg.norm(basis_y)

basis = np.stack([basis_x, basis_y, floor_normal], axis=0)

# Project points into the floor-aligned frame
scene_center = points_valid.mean(axis=0)
coords = (points_valid - scene_center) @ basis.T
coords_xy = coords[:, :2]

# Normalise to a square canvas and accumulate density
extent = np.abs(coords_xy).max()
if extent < 1e-6:
    raise RuntimeError("Degenerate extent for top-down projection")
margin = 1.05
coords_norm = coords_xy / (extent * margin)

resolution = 1024
uv = ((coords_norm + 1.0) * 0.5 * (resolution - 1)).astype(int)
uv = np.clip(uv, 0, resolution - 1)
rows, cols = uv[:, 1], uv[:, 0]

density = np.zeros((resolution, resolution), dtype=np.int32)
np.add.at(density, (rows, cols), 1)

density_vis = np.log1p(density)

# Camera positions and headings in floor frame
extrinsic_batch = extrinsic.squeeze(0).detach()
S = extrinsic_batch.shape[0]
extrinsic_h = torch.zeros((S, 4, 4), dtype=extrinsic_batch.dtype, device=extrinsic_batch.device)
extrinsic_h[:, :3, :4] = extrinsic_batch
extrinsic_h[:, 3, 3] = 1.0
cam_to_world = closed_form_inverse_se3(extrinsic_h).cpu().numpy()

cam_positions = cam_to_world[:, :3, 3]
cam_coords = (cam_positions - scene_center) @ basis.T
cam_norm = cam_coords[:, :2] / (extent * margin)
cam_pix = (cam_norm + 1.0) * 0.5 * (resolution - 1)

forward_vectors = cam_to_world[:, :3, 2]
forward_coords = forward_vectors @ basis.T
forward_xy = forward_coords[:, :2]
forward_norm = np.linalg.norm(forward_xy, axis=1, keepdims=True)
forward_norm[forward_norm < 1e-6] = 1.0
forward_xy_unit = forward_xy / forward_norm
arrow_len = resolution * 0.05
arrow_vec = forward_xy_unit * arrow_len

plt.figure(figsize=(6, 6))
plt.imshow(density_vis, cmap='inferno', origin='lower')
plt.scatter(cam_pix[:, 0], cam_pix[:, 1], c='cyan', s=8, label='Camera centers')
plt.quiver(
    cam_pix[:, 0],
    cam_pix[:, 1],
    arrow_vec[:, 0],
    arrow_vec[:, 1],
    angles='xy',
    scale_units='xy',
    scale=1.0,
    color='cyan',
    width=0.003,
)
plt.title('Top-down density map with camera headings')
plt.axis('off')


## 9. Package outputs for downstream use
Wrap the tensors in a dictionary similar to the demo for reuse.

In [ ]:
predictions = {
    "images": batched_images.squeeze(0).detach().cpu(),
    "pose_enc": pose_enc.detach().cpu(),
    "extrinsic": extrinsic.detach().cpu(),
    "intrinsic": intrinsic.detach().cpu(),
    "depth": depth_map.detach().cpu(),
    "depth_conf": depth_conf.detach().cpu(),
    "world_points": point_map.detach().cpu(),
    "world_points_conf": point_conf.detach().cpu(),
    "track": track_list[-1].detach().cpu(),
    "vis": vis_score.detach().cpu(),
    "conf": conf_score.detach().cpu(),
    "world_points_from_depth": point_map_by_unprojection,
}

for key, value in predictions.items():
    if isinstance(value, torch.Tensor):
        print(f"{key}: {value.shape} ({value.dtype})")
    else:
        print(f"{key}: numpy array with shape {value.shape} and dtype {value.dtype}")


images: torch.Size([41, 3, 294, 518]) (torch.float32)
pose_enc: torch.Size([1, 41, 9]) (torch.float32)
extrinsic: torch.Size([1, 41, 3, 4]) (torch.float32)
intrinsic: torch.Size([1, 41, 3, 3]) (torch.float32)
depth: torch.Size([1, 41, 294, 518, 1]) (torch.float32)
depth_conf: torch.Size([1, 41, 294, 518]) (torch.float32)
world_points: torch.Size([1, 41, 294, 518, 3]) (torch.float32)
world_points_conf: torch.Size([1, 41, 294, 518]) (torch.float32)
track: torch.Size([1, 41, 2, 2]) (torch.float32)
vis: torch.Size([1, 41, 2]) (torch.float32)
conf: torch.Size([1, 41, 2]) (torch.float32)
world_points_from_depth: numpy array with shape (41, 294, 518, 3) and dtype float64


## 10. Output tensor glossary
The table below summarises each tensor returned in `predictions`, its shape convention (batch `B`, frames `S`, height `H`, width `W`, tracked points `N`), and its semantic meaning:returning to earlier sections if you need more detail.

| Key | Shape | Meaning |
| --- | --- | --- |
| `images` | `(S, 3, H, W)` | Preprocessed RGB inputs in `[0, 1]`, per frame.
| `pose_enc` | `(B, S, 9)` | Camera pose encoding (translation, quaternion, FoV) for each frame.
| `extrinsic` | `(B, S, 3, 4)` | Camera-to-world 3×4 SE(3) matrices (OpenCV cam-from-world convention).
| `intrinsic` | `(B, S, 3, 3)` | Pixel intrinsics with principal point centred at `(W/2, H/2)`.
| `depth` | `(B, S, H, W, 1)` | Predicted metric depth per pixel.
| `depth_conf` | `(B, S, H, W)` | Confidence for each depth value (higher is better).
| `world_points` | `(B, S, H, W, 3)` | 3D points from the point-head branch (camera frame).
| `world_points_conf` | `(B, S, H, W)` | Confidence for each point-head prediction.
| `track` | `(B, S, N, 2)` | 2D pixel locations of user-specified tracked points across frames.
| `vis` | `(B, S, N)` | Visibility scores for the tracked points.
| `conf` | `(B, S, N)` | Confidence scores for the tracked point trajectories.
| `world_points_from_depth` | `(S, H, W, 3)` | Depth-unprojected 3D points (NumPy) in world coordinates.
